# Basic Seasonal Elo Rating

This notebook demonstrates the core seasonal rating logic for the Ranking & Rating System.

The **seasonal rating** represents a participant's performance during the current season. It starts from a baseline and changes after every match based on the expected and actual result.

## 1. Seasonal Rating Logic

For every season:

1. Start each participant at **1500**.
2. Calculate the expected score from the current ratings.
3. Compare the expected result with the actual result.
4. Apply the Elo update.
5. Use the new ratings for the next match.

A win against a stronger opponent gives a larger increase than a win against a weaker opponent.

In [ ]:
import pandas as pd

In [ ]:
INITIAL_RATING = 1500
K_FACTOR = 40

def expected_score(player_rating, opponent_rating):
    return 1 / (1 + 10 ** ((opponent_rating - player_rating) / 400))

def update_rating(player_rating, opponent_rating, actual_score, k=K_FACTOR):
    expected = expected_score(player_rating, opponent_rating)
    return player_rating + k * (actual_score - expected)

## 2. One Match Example

Alice and Bob both begin at 1500. Alice wins, so Alice receives a positive update and Bob receives the corresponding negative update.

In [ ]:
alice = 1500
bob = 1500

alice_expected = expected_score(alice, bob)
bob_expected = expected_score(bob, alice)

alice_new = update_rating(alice, bob, 1)
bob_new = update_rating(bob, alice, 0)

print(f"Alice expected score: {alice_expected:.2f}")
print(f"Bob expected score:   {bob_expected:.2f}")
print(f"Alice new rating:     {alice_new:.2f}")
print(f"Bob new rating:       {bob_new:.2f}")

## 3. Process a Complete Season

The matches are processed chronologically. A participant's rating after one match becomes the input for the next match.

In [ ]:
matches = pd.DataFrame([
    ["2026-01-10", "Alice", "Bob", 1],
    ["2026-01-17", "Bob", "Charlie", 1],
    ["2026-01-24", "Alice", "Charlie", 0],
    ["2026-02-01", "Alice", "Bob", 1],
    ["2026-02-08", "Charlie", "Alice", 1],
], columns=["date", "winner", "loser", "winner_score"])

matches["date"] = pd.to_datetime(matches["date"])
matches = matches.sort_values("date")
matches

In [ ]:
ratings = {}
history = []

def get_rating(player):
    return ratings.get(player, INITIAL_RATING)

for _, match in matches.iterrows():
    winner = match["winner"]
    loser = match["loser"]

    winner_before = get_rating(winner)
    loser_before = get_rating(loser)

    winner_expected = expected_score(winner_before, loser_before)
    loser_expected = expected_score(loser_before, winner_before)

    winner_after = winner_before + K_FACTOR * (1 - winner_expected)
    loser_after = loser_before + K_FACTOR * (0 - loser_expected)

    ratings[winner] = winner_after
    ratings[loser] = loser_after

    history.append({
        "date": match["date"],
        "winner": winner,
        "loser": loser,
        "winner_before": round(winner_before, 2),
        "loser_before": round(loser_before, 2),
        "winner_after": round(winner_after, 2),
        "loser_after": round(loser_after, 2),
    })

season_history = pd.DataFrame(history)
season_history

## 4. Seasonal Leaderboard

The final rating after all matches gives the current seasonal leaderboard.

In [ ]:
leaderboard = (
    pd.DataFrame([
        {"player": player, "seasonal_rating": round(rating, 2)}
        for player, rating in ratings.items()
    ])
    .sort_values("seasonal_rating", ascending=False)
    .reset_index(drop=True)
)

leaderboard

## 5. Starting a New Season

A simple seasonal reset can carry part of the previous season's rating into the next season while pulling it toward the 1500 baseline.

With a 25% carry-over:

`new rating = 1500 + 0.25 × (old rating − 1500)`

In [ ]:
CARRY_FRACTION = 0.25

def start_new_season(previous_rating, baseline=INITIAL_RATING, carry_fraction=CARRY_FRACTION):
    return baseline + carry_fraction * (previous_rating - baseline)

new_season = leaderboard.copy()
new_season["seasonal_rating"] = new_season["seasonal_rating"].apply(start_new_season).round(2)
new_season

## 6. Rating Trajectory

Tracking the rating after each match makes it possible to see how current-season form develops over time.

In [ ]:
import matplotlib.pyplot as plt

running = {}
points = []

for _, match in matches.iterrows():
    winner, loser = match["winner"], match["loser"]
    wr = running.get(winner, INITIAL_RATING)
    lr = running.get(loser, INITIAL_RATING)

    we = expected_score(wr, lr)
    le = expected_score(lr, wr)

    running[winner] = wr + K_FACTOR * (1 - we)
    running[loser] = lr + K_FACTOR * (0 - le)

    points.append({"match": len(points) + 1, **running})

trajectory = pd.DataFrame(points).ffill()

for player in ["Alice", "Bob", "Charlie"]:
    plt.plot(trajectory["match"], trajectory[player], marker="o", label=player)

plt.axhline(INITIAL_RATING, linestyle="--", label="Baseline")
plt.xlabel("Match")
plt.ylabel("Seasonal Elo Rating")
plt.title("Seasonal Rating Progression")
plt.legend()
plt.show()

## 7. Summary

**Seasonal Rating = current-season competitive form.**

The foundation is intentionally simple: initialize at 1500, process matches chronologically, calculate expected performance, apply the Elo update, and rank participants by their final seasonal rating.

The other dimensions in the larger project—Overall, Prosperity, and Thinkers—can be layered on later without changing this basic seasonal mechanism.